In [1]:
!pip install -q pymupdf transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 76.0 MB/s eta 0:00:00:00:0100:01


In [4]:
import os, glob
 

PDF_PATH = "/kaggle/input/datasets/sankarraja007/intro-to-algo/Introduction to Algorithms_ (z-library.sk 1lib.sk z-lib.sk).pdf"
print(f"Using PDF: {PDF_PATH}")
 
# ── Chunking config ────────────────────────────────────────────────────────
START_PAGE        = 36     # 0-indexed. Page 29 in the PDF viewer = 28 here.
                           # CLRS 3rd ed: Chapter 1 starts around PDF page 28-30.
                           # Check your copy: open the PDF and find "Chapter 1"
                           # then subtract 1 for 0-indexing.
 
PAGES_PER_CHUNK   = 10     # 10 pages per chunk
NUM_CHUNKS        = 4      # 4 chunks → covers ~40 pages (all of Chapter 1)
QUESTIONS_PER_CHUNK = 25   # 25 × 4 = 100 total questions
 
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
 
# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR        = "/kaggle/working"
BOOK_NAME         = "Introduction to Algorithms - CLRS 3rd Edition"
 
print(f"Config: START_PAGE={START_PAGE}, CHUNKS={NUM_CHUNKS}, "
      f"PAGES_PER_CHUNK={PAGES_PER_CHUNK}, Q_PER_CHUNK={QUESTIONS_PER_CHUNK}")
print(f"Will cover PDF pages {START_PAGE+1} → {START_PAGE + NUM_CHUNKS*PAGES_PER_CHUNK}")


Using PDF: /kaggle/input/datasets/sankarraja007/intro-to-algo/Introduction to Algorithms_ (z-library.sk 1lib.sk z-lib.sk).pdf
Config: START_PAGE=36, CHUNKS=4, PAGES_PER_CHUNK=10, Q_PER_CHUNK=25
Will cover PDF pages 37 → 76


In [5]:
import fitz  # PyMuPDF
 
 
def extract_chunk_text(pdf_path: str, start_page: int, end_page: int) -> str:
    """
    Extract clean body text from [start_page, end_page) (0-indexed).
    Skips headers/footers using vertical position filtering.
    Pseudocode blocks are kept as-is — they're valuable context.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    end_page = min(end_page, total_pages)
 
    full_text = ""
    for page_num in range(start_page, end_page):
        page = doc[page_num]
        page_height = page.rect.height
 
        # get_text("blocks") → each block: (x0, y0, x1, y1, text, block_no, block_type)
        blocks = page.get_text("blocks")
 
        # Filter: skip top 7% (header) and bottom 7% (footer/page-number)
        body_blocks = [
            b[4] for b in blocks
            if b[1] > page_height * 0.07
            and b[3] < page_height * 0.93
            and b[6] == 0          # block_type 0 = text (not image)
            and b[4].strip()       # skip empty blocks
        ]
 
        page_text = "\n".join(body_blocks)
 
        # Light cleanup: collapse 3+ newlines, strip trailing whitespace per line
        import re
        page_text = re.sub(r'\n{3,}', '\n\n', page_text)
        page_text = "\n".join(line.rstrip() for line in page_text.splitlines())
 
        full_text += f"\n\n{'='*60}\n[PDF PAGE {page_num + 1}]\n{'='*60}\n{page_text}"
 
    doc.close()
    return full_text.strip()
 
 
# Quick sanity check — print first 500 chars of chunk 0
sample = extract_chunk_text(PDF_PATH, START_PAGE, START_PAGE + PAGES_PER_CHUNK)
print(f"Sample extraction (first 500 chars):\n{sample[:500]}")
print(f"\nTotal chars in chunk 0: {len(sample)}")


Sample extraction (first 500 chars):
[PDF PAGE 37]
2
Getting Started

This chapter will familiarize you with the framework we shall use throughout the
book to think about the design and analysis of algorithms. It is self-contained, but
it does include several references to material that we introduce in Chapters 3 and 4.
(It also contains several summations, which Appendix A shows how to solve.)

We begin by exam

Total chars in chunk 0: 24585


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
 
print(f"Loading {MODEL_ID} ...")
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")
 
 
def generate_response(system_prompt: str, user_prompt: str, max_new_tokens: int = 6144) -> str:
    """Single call to the model using Qwen chat template."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
 
    # Slice off the input tokens — only return generated part
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


Loading Qwen/Qwen2.5-3B-Instruct ...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [7]:
SYSTEM_PROMPT = """You are an expert computer science professor specializing in algorithms.
Your task is to read a section from "Introduction to Algorithms" (CLRS) and generate
high-quality question-answer pairs for a student study dataset.
 
STRICT OUTPUT FORMAT — every single question MUST use this exact XML structure:
 
<qa_pair>
<question>The full question text here</question>
<answer>
<thinking>
Step-by-step reasoning: explain how you arrive at the answer, referencing concepts from the text.
</thinking>
<final_answer>
A clear, concise, self-contained answer a student can study from.
</final_answer>
</answer>
<difficulty>Easy</difficulty>
<main_topic>DSA</main_topic>
<sub_topic>specific sub-topic (e.g., Loop Invariants, Asymptotic Analysis, Insertion Sort)</sub_topic>
<main_format>Objective</main_format>
<sub_format>MCQ</sub_format>
</qa_pair>
 
RULES:
1. difficulty must be exactly one of: Easy / Medium / Hard
2. main_topic must be exactly one of: DSA / CN / DBMS / OS  (use DSA for algorithms)
3. main_format must be exactly one of: Objective / Subjective
4. sub_format:
   - If main_format=Objective → exactly one of: MCQ / Fill_blank / Assertion / Analytical
   - If main_format=Subjective → must be: Conceptual
5. Generate a BALANCED MIX:
   - Difficulty: ~40% Easy, ~40% Medium, ~20% Hard
   - Format: ~50% Objective (MCQ/Fill_blank/Assertion/Analytical), ~50% Subjective (Conceptual)
6. For MCQ questions, include 4 options (A/B/C/D) inside the <question> tag and state
   the correct option in <final_answer>.
7. Do NOT generate duplicate or trivially similar questions.
8. Do NOT include questions answerable in 1-2 words with no reasoning required.
9. Do NOT add any text outside the <qa_pair>...</qa_pair> blocks.
   No preamble, no "Here are the questions", nothing.
10. Generate EXACTLY {n} qa_pair blocks.
"""
 
 
def build_user_prompt(chunk_text: str, n_questions: int, chunk_idx: int) -> str:
    # Truncate to ~3500 tokens worth of chars to stay within context safely
    # Qwen2.5-3B has 32k context but we need room for output
    max_chars = 6000
    if len(chunk_text) > max_chars:
        chunk_text = chunk_text[:max_chars] + "\n\n[... section continues ...]"
 
    return (
        f"Generate exactly {n_questions} question-answer pairs from the following "
        f"textbook section (Chunk {chunk_idx + 1}).\n\n"
        f"TEXTBOOK SECTION:\n{chunk_text}"
    )


In [8]:
import re
 
 
def safe_get(tag: str, text: str) -> str:
    """Extract content of an XML tag. Returns empty string if not found."""
    m = re.search(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL)
    return m.group(1).strip() if m else ""
 
 
def parse_qa_output(raw: str, chunk_idx: int, start_page: int) -> list[dict]:
    """
    Parse all <qa_pair> blocks from raw model output.
    Returns a list of row dicts ready for a DataFrame.
    """
    pairs = re.findall(r'<qa_pair>(.*?)</qa_pair>', raw, re.DOTALL)
 
    if not pairs:
        print(f"  [WARNING] No <qa_pair> blocks found in chunk {chunk_idx} output.")
        print(f"  Raw output preview:\n{raw[:800]}\n")
        return []
 
    rows = []
    for idx, pair in enumerate(pairs):
        question     = safe_get("question", pair)
        thinking     = safe_get("thinking", pair)
        final_answer = safe_get("final_answer", pair)
        difficulty   = safe_get("difficulty", pair)
        main_topic   = safe_get("main_topic", pair)
        sub_topic    = safe_get("sub_topic", pair)
        main_format  = safe_get("main_format", pair)
        sub_format   = safe_get("sub_format", pair)
 
        # Skip malformed pairs
        if not question or not final_answer:
            print(f"  [SKIP] Pair {idx+1} missing question or final_answer.")
            continue
 
        # Basic value normalization
        difficulty  = difficulty.capitalize()
        main_format = main_format.capitalize()
        if difficulty not in ("Easy", "Medium", "Hard"):
            difficulty = "Medium"
        if main_format not in ("Objective", "Subjective"):
            main_format = "Subjective"
 
        page_start = start_page + chunk_idx * PAGES_PER_CHUNK + 1  # 1-indexed display
        page_end   = page_start + PAGES_PER_CHUNK - 1
 
        rows.append({
            "question":        question,
            "answer":          final_answer,
            "cot_reasoning":   thinking,
            "difficulty":      difficulty,
            "main_topic":      main_topic,
            "sub_topic":       sub_topic,
            "main_format":     main_format,
            "sub_format":      sub_format,
            "question_number": len(rows) + 1,
            "chunk_index":     chunk_idx + 1,
            "batch_number":    1,
            "book_name":       BOOK_NAME,
            "page_number_range": f"{page_start} - {page_end}",
        })
 
    return rows
 

In [ ]:
import pandas as pd
import time
 
all_rows   = []
system_msg = SYSTEM_PROMPT.format(n=QUESTIONS_PER_CHUNK)
 
for chunk_idx in range(NUM_CHUNKS):
    chunk_start = START_PAGE + chunk_idx * PAGES_PER_CHUNK
    chunk_end   = chunk_start + PAGES_PER_CHUNK
 
    print(f"\n{'━'*60}")
    print(f"CHUNK {chunk_idx + 1}/{NUM_CHUNKS}  |  PDF pages {chunk_start+1}–{chunk_end}")
    print(f"{'━'*60}")
 
    # ── Step A: Extract text ───────────────────────────────────────────────
    print("  Extracting text from PDF...")
    chunk_text = extract_chunk_text(PDF_PATH, chunk_start, chunk_end)
    print(f"  Extracted {len(chunk_text)} chars.")
 
    # ── Step B: Generate QA pairs ──────────────────────────────────────────
    user_msg = build_user_prompt(chunk_text, QUESTIONS_PER_CHUNK, chunk_idx)
    print(f"  Generating {QUESTIONS_PER_CHUNK} QA pairs (this takes ~2-4 min)...")
 
    t0 = time.time()
    try:
        raw_output = generate_response(system_msg, user_msg, max_new_tokens=6144)
    except Exception as e:
        print(f"  [ERROR] Model generation failed: {e}")
        print("  Saving progress so far and skipping this chunk.")
        if all_rows:
            pd.DataFrame(all_rows).to_csv(
                os.path.join(OUTPUT_DIR, "clrs_dataset_partial.csv"), index=False
            )
        continue
 
    elapsed = time.time() - t0
    print(f"  Generation done in {elapsed:.1f}s. Output length: {len(raw_output)} chars.")
 
    # ── Step C: Parse ──────────────────────────────────────────────────────
    print("  Parsing XML output...")
    rows = parse_qa_output(raw_output, chunk_idx, START_PAGE)
    print(f"  Parsed {len(rows)} valid pairs.")

 
    # ── Step E: Save chunk CSV immediately ────────────────────────────────
    if rows:
        chunk_csv_path = os.path.join(OUTPUT_DIR, f"clrs_chunk_{chunk_idx + 1}.csv")
        pd.DataFrame(rows).to_csv(chunk_csv_path, index=False)
        print(f"  ✓ Saved chunk CSV → {chunk_csv_path}")
    else:
        print(f"  [WARNING] No rows to save for chunk {chunk_idx + 1}.")
 
    all_rows.extend(rows)
 
    # ── Step F: Running total CSV ──────────────────────────────────────────
    if all_rows:
        total_csv_path = os.path.join(OUTPUT_DIR, "clrs_dataset_all_chunks.csv")
        pd.DataFrame(all_rows).to_csv(total_csv_path, index=False)
        print(f"  ✓ Updated combined CSV → {total_csv_path}  ({len(all_rows)} total rows)")
 
print(f"\n{'='*60}")
print(f"DONE.  Total QA pairs generated: {len(all_rows)}")
print(f"Individual chunk CSVs: clrs_chunk_1.csv … clrs_chunk_{NUM_CHUNKS}.csv")
print(f"Combined CSV:          clrs_dataset_all_chunks.csv")
print(f"All files in:          {OUTPUT_DIR}")
print(f"{'='*60}")



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CHUNK 1/4  |  PDF pages 37–46
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Extracting text from PDF...
  Extracted 24585 chars.
  Generating 25 QA pairs (this takes ~2-4 min)...


In [ ]:
if all_rows:
    df = pd.DataFrame(all_rows)
 
    print("\nDataset Shape")
    print(f"  Rows: {len(df)}  |  Columns: {list(df.columns)}")
 
    print("\nDifficulty Distribution")
    print(df["difficulty"].value_counts().to_string())
 
    print("\nMain Format Distribution")
    print(df["main_format"].value_counts().to_string())
 
    print("\nSub Format Distribution")
    print(df["sub_format"].value_counts().to_string())
 
    print("\nSub Topics Covered")
    print(df["sub_topic"].value_counts().head(15).to_string())
 
    print("\nSample Row")
    sample_row = df.iloc[0]
    print(f"  Q: {sample_row['question'][:120]}...")
    print(f"  A: {sample_row['answer'][:120]}...")
    print(f"  Difficulty: {sample_row['difficulty']}  |  Format: {sample_row['main_format']} / {sample_row['sub_format']}")
    print(f"  Sub-topic: {sample_row['sub_topic']}")